In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("OlistExploration").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/06 20:56:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
# 2. Define the datasets dictionary
datasets = {
    "orders": "olist_orders_dataset.csv",
    "items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "products": "olist_products_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "translation": "product_category_name_translation.csv"
}

# 3. Set the path (relative to where this notebook is saved)
base_path = "data/raw/" 

# 4. Load the data
dfs = {}
for name, file_name in datasets.items():
    print(f"Loading {file_name}...")
    dfs[name] = spark.read.csv(f"{base_path}{file_name}", header=True, inferSchema=True)
    
print("All files loaded successfully!")

Loading olist_orders_dataset.csv...


Loading olist_order_items_dataset.csv...


Loading olist_order_payments_dataset.csv...
Loading olist_order_reviews_dataset.csv...


Loading olist_customers_dataset.csv...
Loading olist_sellers_dataset.csv...
Loading olist_products_dataset.csv...
Loading olist_geolocation_dataset.csv...


Loading product_category_name_translation.csv...
All files loaded successfully!


In [5]:
from pyspark.sql.functions import *
# Order and products schema
dfs["orders"].printSchema()
dfs["products"].printSchema()


root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)

root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)



In [6]:
#Counting null value in orders and in products
dfs["orders"].select([
    count(when(col(c).isNull(), c)).alias(c) for c in dfs["orders"].columns
]).show(vertical=True)

dfs["products"].select([
    count(when(col(c).isNull(), c)).alias(c) for c in dfs["products"].columns
]).show(vertical=True)

-RECORD 0-----------------------------
 order_id                      | 0    
 customer_id                   | 0    
 order_status                  | 0    
 order_purchase_timestamp      | 0    
 order_approved_at             | 160  
 order_delivered_carrier_date  | 1783 
 order_delivered_customer_date | 2965 
 order_estimated_delivery_date | 0    



-RECORD 0-------------------------
 product_id                 | 0   
 product_category_name      | 610 
 product_name_lenght        | 610 
 product_description_lenght | 610 
 product_photos_qty         | 610 
 product_weight_g           | 2   
 product_length_cm          | 2   
 product_height_cm          | 2   
 product_width_cm           | 2   



In [8]:
#Checking for duplicates
dfs["orders"].groupBy("order_id").count().filter("count > 1").show()

dfs["reviews"].groupBy("review_id").count().filter("count > 1").show()

+--------+-----+
|order_id|count|
+--------+-----+
+--------+-----+



+--------------------+-----+
|           review_id|count|
+--------------------+-----+
|f144ac19984746532...|    2|
|e4f5fbbbf2fa8f259...|    2|
|f6e05a3f6dbc3bdb1...|    2|
|95cfb15ad3477fec5...|    2|
|dbcd301b59c7e85e6...|    2|
|ef60daa08e71f279b...|    2|
|4812973f7fb59a488...|    2|
|f45e3373e177f5694...|    2|
|9d7b240dff5539983...|    2|
|d8a302244615111a0...|    2|
|fde5986d35c89aa1b...|    2|
|89d6214895235bb95...|    2|
|77d427beb34a26b48...|    2|
|muito feliz pelo ...|    2|
|Bem antes do praz...|    2|
|29aeeca2ed21aedc5...|    2|
|4687cd3dc8cbaa6b8...|    2|
|       RECOMENDO!!!"|    2|
|0ad57b9e68d4d3481...|    2|
|561fea58ddff8654c...|    2|
+--------------------+-----+
only showing top 20 rows


In [9]:
#Multiline fix
dfs["reviews"] = spark.read.csv(
    f"{base_path}olist_order_reviews_dataset.csv", 
    header=True, 
    inferSchema=True, 
    multiLine=True, 
    escape='"'
)

# Re-run the duplicate check
dfs["reviews"].groupBy("review_id").count().filter("count > 1").show()

+--------------------+-----+
|           review_id|count|
+--------------------+-----+
|f144ac19984746532...|    2|
|e4f5fbbbf2fa8f259...|    2|
|868c4827b9ab72723...|    2|
|a8ffb30c74397c7cf...|    2|
|f6e05a3f6dbc3bdb1...|    2|
|c9a406d3076559e35...|    2|
|95cfb15ad3477fec5...|    2|
|dbcd301b59c7e85e6...|    2|
|9a07cb8136a1792e7...|    2|
|dd43a798c9e8baf7b...|    2|
|ef60daa08e71f279b...|    2|
|4812973f7fb59a488...|    2|
|f45e3373e177f5694...|    2|
|9d7b240dff5539983...|    2|
|4e9a141db068cf685...|    2|
|c1dd7c6a2154caaf1...|    2|
|bee3c76b6017ec9de...|    2|
|1961cafe1d1cbd4cc...|    2|
|d8a302244615111a0...|    2|
|fde5986d35c89aa1b...|    2|
+--------------------+-----+
only showing top 20 rows
